In [ ]:
import pandas as pd
import requests
import os
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.stattools import durbin_watson
import numpy as np
import scipy.stats as stats
from linearmodels.panel import PanelOLS
from linearmodels.panel import RandomEffects

In [ ]:
# run this cell if you don't have linearmodels package
!pip install linearmodels

# Topic 2: Panel Data

This session will emphasize panel regression using both continuous and classification problems as examples.
The discussion will potentially cover the following areas:
* Why using panel regression? heterogeneity, endogeneity, leveraging both cross-sectional and time-series dimensions for efficiency and dynamic.
* Fix effect v.s. Random effect
* Difference in Difference analysis
* Typical tests, such as Hausman test

## Load Data

In [ ]:
# Define get_example_ds function to download dataset to the current directory
def get_example_ds(stata_url,local_filename):
    try:
        # Download the file
        response = requests.get(stata_url, timeout=10)
        response.raise_for_status()  # Raise error for bad status codes

        # Save locally
        with open(local_filename, "wb") as f:
            f.write(response.content)
        print(f"Downloaded dataset to {os.path.abspath(local_filename)}")
    except requests.exceptions.RequestException as e:
        print(f"Error downloading dataset: {e}")
    except (ValueError, OSError) as e:
        print(f"Error reading dataset: {e}")

The nlswork dataset is extracted from the National Longitudinal Survey (NLS) of Young Women.It records annual data collected between 1968 and 1988 (with some gaps), including 28,534 person-year records across 4,711 unique individuals

* `idcode`: NLS ID; a unique identifier tracking the same individual across multiple years.
* `year`: Interview year; represents the calendar year the interview took place (ranges from 1968 to 1988).
* `birth_yr`: Birth year; the calendar year the respondent was born.
* `age`: Age in current year; the exact age of the woman at the time of that year's interview.
* `race`: Race category; coded categorically as 1 = white, 2 = black, and 3 = other.
* `grade`: Current grade completed; highest integer grade level achieved by the respondent.
* `collgrad`: College graduate indicator; a binary marker coded as 1 if the respondent has graduated from college, and 0 otherwise.
* `msp`: Married, spouse present; a binary marker coded as 1 if the respondent is married and living with their spouse.
* `nev_mar`: Never married indicator; a binary marker coded as 1 if the woman has never been married.
* `not_smsa`: Outside SMSA; a binary marker coded as 1 if the respondent lives outside a Standard Metropolitan Statistical Area (rural/non-urban zone).
* `c_city`: Central city indicator; a binary marker coded as 1 if the respondent lives within the urban core of an SMSA.
* `south`: Southern region indicator; a binary marker coded as 1 if the respondent resides in the Southern United States.
* `ind_code`: Industry of employment; a numerical code identifying the sector of the economy they work in.
* `occ_code`: Occupation code; a numerical code identifying the worker's specific job type or role.
* `union`: Union membership; a binary marker coded as 1 if the respondent's current job is unionized.
* `wks_ue`: Weeks unemployed last year; the total count of weeks spent out of work and looking for a job in the preceding year.
* `wks_work`: Weeks worked last year; the total count of weeks the respondent was actively employed in the preceding year.
* `ttl_exp`: Total work experience; cumulative years of work experience accumulated over their lifetime up to that point.
* `tenure`: Job tenure; the continuous number of years the respondent has been working with their current employer.
* `hours`: Usual hours worked; the standard number of hours the respondent works per week.
* `ln_wage`: Log of wages; the natural logarithm of the respondent's real wage, adjusted by the GNP deflator.

In [ ]:
# Example: Download Stata's "nlswork.dta" dataset - Longevity
stata_url = "https://www.stata-press.com/data/r18/nlswork.dta"  # Change r18 to your Stata release if needed
local_filename = "nlswork.dta"

# Download dataset
get_example_ds(stata_url,local_filename)

# Load into pandas DataFrame
df_nls = pd.read_stata(local_filename)

# Preview the dataframe
df_nls.head()

In [ ]:
print('Number of interviewees: ',len(df_nls['idcode'].unique()))

In [ ]:
df_nls.describe()

In [ ]:
df_nls.loc[df_nls['idcode']==4501,['year','ln_wage']].set_index('year').plot()
df_nls.loc[df_nls['idcode']==1397,['year','ln_wage']].set_index('year').plot()
df_nls.loc[df_nls['idcode']==1,['year','ln_wage']].set_index('year').plot()

In [ ]:
df_nls.groupby(['year'])['ln_wage'].mean().plot()

## OLS

In [ ]:
col_y = 'ln_wage'
list_col_x = ['grade', 'ttl_exp', 'tenure', 'union', 'south']
df_data=df_nls[[col_y]+list_col_x]
equation = 'ln_wage ~ grade + ttl_exp + ttl_exp:ttl_exp+ tenure + union + south'
model_ols = smf.ols(equation, data=df_data).fit()
print(model_ols.summary())

**Autocorrelation Test**

In [ ]:
# Pass the model residuals into the durbin_watson function
dw_value = durbin_watson(model_ols.resid)
# Print the result
print(f"Durbin-Watson statistic: {dw_value:.4f}")

**Question**: How to explain the OLS regression results?

## Fixed-Effects (FE) Model

First, we introduce one fixed effect and control for all unobserved time-invariant individual characteristics (like innate ability, family background, or motivation) that might affect both education/experience and wages.

**Question**: try a different equation using the same individual fixed effect `idcode` but different X variables. Can you improve the model performance? What variables should NOT be included to this regression? 

In [ ]:
col_y = 'ln_wage'
list_col_x = ['grade', 'ttl_exp', 'tenure', 'union', 'south', 'idcode']
df_data=df_nls[[col_y]+list_col_x]
equation = 'ln_wage ~ grade + ttl_exp + tenure + union + south + C(idcode) -1'
model_fe1 = smf.ols(equation, data=df_data).fit()
print(model_fe1.summary())

In [ ]:
# Pass the model residuals into the durbin_watson function
dw_value = durbin_watson(model_fe1.resid)
# Print the result
print(f"Durbin-Watson statistic: {dw_value:.4f}")

In [ ]:
# Calculate model residuals
residuals = pd.DataFrame(model_fe1.resid)
# Regress the residuals on their 1-period lag
lagged_resids = residuals.shift(1)
combined = sm.add_constant(pd.concat([residuals, lagged_resids], axis=1).dropna())
combined.columns = ['const', 'residual', 'lagged_residual']
test_model = sm.OLS(combined['residual'], combined[['const', 'lagged_residual']]).fit()
print(test_model.summary())

Next, we introduce 2 fixed effect, individual fixed effect and time fixed effect. Then estimate the model using `linearmodels` package `PanelOLS` method.

**Question**: What does the time fixed effect control for?

**Question**: How to interpret different R-squared statistics?

In [ ]:
from linearmodels.panel import PanelOLS

In [ ]:
df_data=df_nls.set_index(['idcode','year']).copy()

In [ ]:
model_fe2 = PanelOLS.from_formula("ln_wage ~ ttl_exp + tenure + south + union + EntityEffects + TimeEffects", data=df_data)
results_fe2 = model_fe2.fit(cov_type="clustered", cluster_entity=True, cluster_time=True)

print(results_fe2.summary)

## Random-Effects (RE) Model

If the unobserved individual differences are completely uncorrelated with independent variables. Random effect model would work, 
* Time-Invariant Variables: Unlike a Fixed Effects model (PanelOLS with EntityEffects), a Random Effects model allows you to include variables that do not change over time for an individual (like race or fixed education levels).
 
* The Intercept: Always include the intercept (1) in a Random Effects formula to anchor the overall mean of your dependent variable.

In [ ]:
df_data = df_nls.set_index(['idcode', 'year']).copy()

formula = "ln_wage ~ 1 + ttl_exp + tenure + south + union"
model_re = RandomEffects.from_formula(formula, data=df_data)
results_re = model_re.fit(cov_type='clustered', cluster_entity=True)

print(results_re.summary)

**Question**: Let's try to include Time-Invariant Variables. Which variables would you choose?

**Question**: How would you interpret the coefficient of the `union` variable?

In [ ]:
df_data = df_nls.set_index(['idcode', 'year']).copy()

formula = "ln_wage ~ 1 + ttl_exp + tenure + south + union + nev_mar"
model_re = RandomEffects.from_formula(formula, data=df_data)
results_re = model_re.fit(cov_type='clustered', cluster_entity=True)

print(results_re.summary)

## Hausman Test - Which Model is Better?

Run a Hausman specification test to formally decide between the Fixed-Effects and Random-Effects models.

In [ ]:
df_data = df_nls.set_index(["idcode", "year"]).copy()

# Do NOT include purely time-invariant variables (like race) in the comparison,
# as they will automatically drop out of the FE model and cause dimension mismatch.
variables = ["ttl_exp", "tenure", "south","union"]
formula_fe = "ln_wage ~ " + " + ".join(variables) + " + EntityEffects"
formula_re = "ln_wage ~ 1 + " + " + ".join(variables)

# Use standard homoskedastic errors ('unadjusted') for the classic Hausman test
fe_model = PanelOLS.from_formula(formula_fe, data=df_data).fit(cov_type="unadjusted")
re_model = RandomEffects.from_formula(formula_re, data=df_data).fit(cov_type="unadjusted")

b_fe = fe_model.params[variables]
b_re = re_model.params[variables]

v_fe = fe_model.cov[variables].loc[variables]
v_re = re_model.cov[variables].loc[variables]

# Compute the Hausman Test Statistic
b_diff = np.atleast_2d(b_fe - b_re).T
v_diff = v_fe - v_re

# Chi-squared statistic: (b_fe - b_re)' * inv(V_fe - V_re) * (b_fe - b_re)
chi2 = float(np.dot(b_diff.T, np.dot(np.linalg.inv(v_diff), b_diff))[0][0])
df_degrees = len(variables)
p_value = 1 - stats.chi2.cdf(chi2, df_degrees)

print("--- MODEL COMPARISON ---")
comparison = pd.DataFrame({"Fixed Effects": b_fe, "Random Effects": b_re})
comparison["Difference"] = comparison["Fixed Effects"] - comparison[
    "Random Effects"
]
print(comparison)

print("\n--- HAUSMAN TEST RESULTS ---")
print(f"Chi-Squared Statistic: {chi2:.4f}")
print(f"Degrees of Freedom:    {df_degrees}")
print(f"P-value:               {p_value:.4f}")